1. 여기어때 사이트에 접속한다
2. 인기 추천 숙소이 있는 크롤링 대상을 확인한다
3. 상단에 나와있는 top4개의 숙소 정보를 확인한다
4. 카테고리, 숙소명, 숙소위치, 평점, 평가인원수, 원래금액, 할인금액 순으로 정보를 확인한다
5. 해당 데이터를 mysql로 가져온다

In [21]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys 
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import csv

URL = "https://www.yeogi.com/"

service = Service(ChromeDriverManager().install())
options = Options()

options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

driver = webdriver.Chrome(service=service, options=options)
driver.get(URL)
time.sleep(2)

results = list()

for _ in range(1) :
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

    products = driver.find_elements(By.CSS_SELECTOR, "a.css-1plj9tt")[:4]

    for product in products :
        try :
            category = product.find_element(By.CSS_SELECTOR, ".css-zh8h4t").text.strip()
            title = product.find_element(By.CSS_SELECTOR, ".css-1amprqr").text.strip()
            location = product.find_element(By.CSS_SELECTOR, ".css-6a9mcq").text.strip()
            rating = product.find_element(By.CSS_SELECTOR, ".css-ry30z7").text.strip()
            people = product.find_element(By.CSS_SELECTOR, ".css-144z61f").text.strip()
            price = product.find_element(By.CSS_SELECTOR, ".css-1ezkn8h").text.strip()
            dis_price = product.find_element(By.CSS_SELECTOR, ".css-9vsehz").text.strip()
    
            results.append({
                # "No": index + 1, 
                "카테고리": category,
                "숙소명": title,
                "숙소위치": location,
                "평점": rating,
                "평가인원": people,
                "상품가격": price,
                "할인가격": dis_price
            })
    
        except Exception as e :
            print("데이터 크롤링 실패 :", e)
            continue



df = pd.DataFrame(results).head(4)

driver.quit()

df

,카테고리,숙소명,숙소위치,평점,평가인원,상품가격,할인가격
0,블랙 · 특급 · 호텔,★당일특가★ 세인트존스 호텔,강릉시\n강릉 강문해변 앞,9.2,"10,789명 평가","231,000원","219,450"
1,블랙 · 5성급 · 호텔,힐튼 경주,경주시\n보문관광단지 부근,9.4,"9,324명 평가","399,300원","141,000"
2,모텔,길동 MARI-마리,길동역 도보 3분,9.3,"5,616명 평가","50,000원","45,000"
3,모텔,구월동 구월호텔 九,인천터미널역 도보 14분,9.4,"14,469명 평가","44,400원","39,900"


In [19]:
import pymysql

MYSQL_HOST = "127.0.0.1"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PW = "ksdir8558"
MYSQL_DB = "yeogi_db"
MYSQL_CHARSET = "utf8mb4"

rows = []

for r in results : 
    category = r.get("카테고리")
    title = r.get("숙소명")
    location = r.get("숙소위치")
    rating = r.get("평점")
    people = r.get("평가인원")
    price = r.get("상품가격")
    dis_price = r.get("할인가격")

    rows.append((category, title, location, rating, people, price, dis_price))

print("DB INSERT 대상 row 수:", len(rows))

conn = pymysql.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PW,
    database=MYSQL_DB,
    charset=MYSQL_CHARSET
)

try :
    with conn.cursor() as cur :
        sql = """
        INSERT INTO yeogi_product_top4 (category, title, location, rating, people, price, dis_price) 
        VALUES (%s, %s, %s, %s, %s, %s, %s);
        """
        
        for row in rows :
            cur.execute(sql, row)

    conn.commit()
    print("MYSQL 저장완료!")
        
except Exception as e :
    conn.rollback()
    print("😥저장실패: ", e)

finally :
    conn.close()

DB INSERT 대상 row 수: 4
MYSQL 저장완료!
